# Bike Web Scraping

This notebook scrapes bike listings from Flipkart and exports a cleaned dataset.

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re


In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36',
    'accept-encoding': 'gzip, deflate, br, zstd',
    'content-type': 'text/plain;charset=UTF-8',
    'accept-language': 'en-US,en;q=0.9'
}


In [ ]:
all_names = []
all_specs = []
all_prices = []
all_ratings = []

for page in range(1, 30):
    url = f'https://www.flipkart.com/search?q=bikes&page={page}'
    response = requests.get(url, headers=headers, timeout=20)
    print(f'Page {page}: {response.status_code}')

    if response.status_code != 200:
        continue

    soup = BeautifulSoup(response.text, 'html.parser')

    names = soup.find_all('div', class_='RG5Slk')
    specs = soup.find_all('div', class_='CMXw7N')
    prices = soup.find_all('div', class_='hZ3P6w DeU9vF')
    ratings = soup.find_all('div', class_='MKiFS6')

    for item in names:
        text = item.get_text(' ', strip=True)
        if text:
            all_names.append(text)

    for item in specs:
        text = item.get_text(' ', strip=True)
        if text:
            all_specs.append(text)

    for item in prices:
        text = item.get_text(' ', strip=True)
        if text:
            all_prices.append(text)

    for item in ratings:
        text = item.get_text(' ', strip=True)
        if text:
            all_ratings.append(text)


In [ ]:
min_len = min(len(all_names), len(all_specs), len(all_prices), len(all_ratings))
all_names = all_names[:min_len]
all_specs = all_specs[:min_len]
all_prices = all_prices[:min_len]
all_ratings = all_ratings[:min_len]

df = pd.DataFrame({
    'Product_Name': all_names,
    'Specifications': all_specs,
    'Price': all_prices,
    'Rating': all_ratings
})

df.head()


In [ ]:
def clean_price(value):
    value = str(value)
    value = value.replace('₹', '').replace(',', '').replace(' ', '')
    return float(value) if value else None

df['Price'] = df['Price'].apply(clean_price)
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
df.head()


In [ ]:
df_non_electric = df[df['Product_Name'].str.contains('electric', case=False, na=False) == False].copy()
df_non_electric.head()


In [ ]:
df_non_electric.to_csv(
    'Clean_Bike_Dataset.csv',
    index=False
)
